# Tu copia personal · Aula UTP

En Colab elige **Archivo → Guardar una copia en Drive** y trabaja en esa copia. No necesitas activar recursos de pago ni GPU.

**No hace falta ejecutar todo el cuaderno.** Ejecuta las dos celdas de preparación y luego las celdas de tu encuentro, de arriba abajo dentro de ese recorrido. La ruta de abajo indica dónde empezar y terminar. Cada bloque principal crea sus propios ejemplos.

Una celda es un recuadro de explicación o de código. Para ejecutar una de código, pulsa su triángulo y espera a que termine. Leer un gráfico no vuelve a ejecutar la celda. Las comprobaciones son voluntarias, sin entregas ni notas.

# Técnicas de aprendizaje profundo · Laboratorio explicado UTP

Este cuaderno acompaña los cuatro encuentros de Deep Learning. Primero sigue un ejemplo pequeño; después ejecuta el código y explica qué cambió. Los datos de productos y fincas son inventados; los dígitos son imágenes pequeñas incluidas en scikit-learn. Ninguno valida una herramienta para inspección agrícola real.

**Lo que verás en cada bloque:** situación → palabras necesarias → entrada y respuesta → gráfico → una comprobación pequeña. Los números del cuaderno no tienen por qué coincidir con los dibujos del aula: se indica cuándo cambia el conjunto de ejemplos.

Los bloques **02–08 son repaso de ML**, el texto del 11 es una ampliación y el 13 es una ampliación de generación. No forman parte del recorrido principal de estos encuentros. Están desactivados por defecto. Tampoco es necesario abrir proyectos externos de modelos grandes.

**Regla para comparar:** si eliges tasas, épocas u otras opciones mirando un conjunto reservado, estás usando ese conjunto para elegir. Antes de afirmar un resultado final necesitarías otros casos que no hayas mirado para elegir.


## Tu ruta por encuentro · sin recorrer temas ajenos

Abre el índice lateral de Colab para buscar los títulos. Siempre empieza con **Preparación 1 y Preparación 2**.

- **DL · Encuentro 1, 6 de noviembre:** bloque 01, sus dos celdas; después bloque 09, sus dos celdas. Seguimos una neurona, activaciones, un ajuste pequeño y una red que sí aprende.
- **DL · Encuentro 2, 7 de noviembre:** bloque 10, primero CNN y después la ilustración de tareas. Para transferencia, continúa en el cuaderno **YOLO, secciones 1–6**: cargar pesos, comprobar categorías y producir una respuesta. La CNN reconoce dígitos; no aprende a localizar frutos.
- **DL · Encuentro 3, 13 de noviembre:** en el bloque 11 ejecuta solo la sección **11A · Pronosticar la siguiente lectura con LSTM**. Después bloque 12: atención y Transformer. No necesitas ejecutar la celda de textos, TF-IDF o SVD. El ejemplo de atención mezcla dos cantidades; el Transformer responde una pregunta diferente sobre el orden.
- **DL · Encuentro 4, 14 de noviembre:** en el cuaderno YOLO sigue **secciones 7–11 y 13** para registro, filtros y evaluación con cajas anotadas. Después vuelve aquí al bloque 14 para comparar decisiones de revisión. No es necesario el bloque 08 ni SHAP.

**Si cambias un dato:** ejecuta otra vez la celda que lo define y las posteriores del mismo bloque. Si Colab se reinicia, las variables desaparecen: vuelve a ejecutar las dos preparaciones y tu bloque.

**CPU:** usa el entorno estándar con procesador; no actives un acelerador de hardware para completar estas prácticas. Estos ejemplos no exigen contratar Colab Pro, una API ni recursos de pago. Si se corta la sesión, conserva la copia y retoma con CPU.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


## 01. Una neurona y un ajuste: entiende una cuenta pequeña

**La situación:** imaginamos dos medidas de un grano: uniformidad de color 0,6 y parte manchada 0,2. Ambas usan una escala de 0 a 1; 0,6 representa 60 % de esa escala, no 60 granos. Aquí las escogimos para explicar una cuenta: no se midieron de una foto.

**Palabras antes del código:** una entrada es el número recibido; un peso decide cuánto aporta multiplicándolo; el sesgo es otro número que se suma. Estos pesos fueron fijados a mano. Una neurona real aprende sus ajustes con ejemplos; esta primera cuenta no aprende todavía.

Primero mira las contribuciones de color y manchas. Después ejecuta la sección **01B · Corregir un ajuste paso a paso**: es **otro ejemplo**, un único ajuste que queremos acercar a 3. No es el peso del grano anterior.

**Antes de ejecutar:** calcula color: 0,6 × 2 = 1,2; manchas: 0,2 × (−1) = −0,2; sesgo: −0,5. El total debe ser 0,5. Si el peso de manchas fuera −2, su aporte sería −0,4 y el total 0,3.


### Cómo leer la cuenta y el ajuste

En las barras de la neurona, subir significa sumar y bajar significa restar. **0,5 es una suma interna**, todavía no una respuesta fiable de “sano” o “defectuoso”. En el aula se usa otra neurona pequeña con entradas 2 y 1 y ReLU; aquí se usan 0,6 y 0,2 y más adelante sigmoide. Compara el procedimiento, no esperes cifras idénticas.

En el segundo ejemplo llamamos **pérdida** al número que mide qué tan lejos está el ajuste w del objetivo 3. Desde −2 hay una distancia de 5; al elevarla al cuadrado obtenemos 25. La pendiente indica cómo cambia esa pérdida: aquí se calcula como 2 × (w−3). La **tasa** es el tamaño elegido para corregir. Se resta tasa × pendiente al ajuste.

**Tip:** empieza con TASA=0.15, sin cambiar nada más. Si luego pruebas 1.1, observa los puntos: pueden alejarse del objetivo. Más grande no significa mejor. Ese valor 3 es un objetivo inventado para ver la operación, no un peso aprendido del café.


### 01A · Calcular los aportes de una neurona

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
entradas = np.array([.6,.2]); pesos = np.array([2.,-1.]); sesgo = -.5
contribuciones = entradas * pesos
z = entradas @ pesos + sesgo
print('Contribuciones:',contribuciones,'· sesgo:',sesgo,'· suma z:',z)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(['Color × peso','Manchas × peso','Sesgo'],[*contribuciones,sesgo],color=[AZUL,VERDE,GRIS])
ax.axhline(0,color='black',lw=.7)
ax.set(title=f'La suma de estas contribuciones es z = {z:.1f}',ylabel='Aporte a la suma interna')
mostrar('01-neurona')


### Después de ejecutar: sigue las contribuciones

Comprueba 1,2 − 0,2 − 0,5 = 0,5. No confundas la barra negativa con un número negativo de manchas: es la contribución del peso elegido. La figura muestra cómo se forma una salida; no mide todavía si esa salida acierta.

### 01B · Corregir un ajuste paso a paso

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Pérdida L(w) = (w−3)^2. Su derivada es 2(w−3).
TASA = .15 # prueba .02 o 1.1 y compara, no siempre converge
w = -2.; camino = [w]
for paso in range(12):
    w = w - TASA*2*(w-3)
    camino.append(w)
curva=np.linspace(-3,5,120)
plt.plot(curva,(curva-3)**2,color=AZUL);plt.plot(camino,(np.array(camino)-3)**2,'o-',color=VERDE)
plt.title('Descenso del gradiente: pasos hacia menor error');plt.xlabel('Peso w');plt.ylabel('Pérdida (w−3)²');mostrar('01-gradiente')
print('Peso final:',round(w,3),'· error final:',round((w-3)**2,4))


### Después de ejecutar: ¿se acerca o se aleja?

Sigue los puntos desde el inicio. La altura es pérdida, no exactitud ni dinero. Si cambias la tasa, vuelve a ejecutar desde el inicio para comparar ambos recorridos desde −2. Un punto cercano a w=3 es menor pérdida en esta cuenta elegida, no una red validada.

## 02. Python: una herramienta por tarea
 **El problema:** Tenemos rendimientos de cuatro lotes y queremos resumirlos sin calcular a mano.

 **Vamos a hacer:** Usar lista, función, NumPy y una tabla de Pandas.

 **Antes de ejecutar:** Añade un quinto lote de rendimiento 50. ¿Qué ocurrirá con la media?

 **Cómo comprobarlo:** La media original es 2,5; al agregar 50 aumenta mucho. Un valor extremo puede distorsionar el resumen.


### Entiende la operación · Fila, columna y resumen
Los cuatro rendimientos son ilustrativos y no tienen unidades físicas definidas en este bloque. Promedio=(1+2+3+4)/4=2,5. Una barra representa una fila; su altura es el valor, no el promedio. Una función recibe valores y devuelve un resultado; NumPy repite la operación sobre un arreglo y pandas conserva nombres de filas y columnas.

In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_REPASO_ML:
    rendimientos = [1,2,3,4]
    def promedio(valores):
        return sum(valores)/len(valores)
    tabla = pd.DataFrame({'Lote':['A','B','C','D'],'Rendimiento':rendimientos})
    display(tabla)
    print('Con Python:',promedio(rendimientos),'· con NumPy:',np.mean(rendimientos))
    tabla.plot.bar(x='Lote',y='Rendimiento',color=AZUL,legend=False)
    plt.ylabel('Rendimiento ilustrativo');plt.title('Cada fila es un lote; cada columna, una variable');mostrar('02-python')
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


## 03. Datos y EDA: mirar antes de modelar
 **El problema:** Faltan lecturas de humedad y las variables tienen unidades muy distintas.

 **Vamos a hacer:** Crear un CSV, visualizar, separar entrenamiento/prueba e imputar y escalar dentro de un Pipeline.

 **Antes de ejecutar:** ¿Qué significa humedad = 0,8 después de escalar? ¿Es 80 % de humedad física?

 **Cómo comprobarlo:** No: es una posición dentro del rango aprendido en entrenamiento. Los parámetros de limpieza se aprenden sin mirar la prueba.


### Entiende la operación · Aprender también la preparación
Los valores faltantes se introducen a propósito. Primero se reserva prueba; luego la mediana y los extremos de MinMaxScaler se aprenden solo con entrenamiento. Mediana es el centro ordenado; MinMax usa (x−mín)/(máx−mín). Un valor de prueba puede quedar fuera de [0,1] si supera los extremos aprendidos. Es una transformación, no una probabilidad.

In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_REPASO_ML:
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.linear_model import LinearRegression
    X,y=datos_fincas()
    X.loc[X.index[::13],'Humedad_pct']=np.nan
    archivo=CARPETA/'fincas_simuladas.csv'
    X.assign(Rendimiento_t_ha=y).to_csv(archivo,index=False)
    leido=pd.read_csv(archivo);display(leido.head())
    print('Faltantes por columna:\n',X.isna().sum())
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEMILLA)
    modelo=make_pipeline(SimpleImputer(strategy='median'),MinMaxScaler(),LinearRegression())
    modelo.fit(Xtr,ytr);pred=modelo.predict(Xte)
    print('MAE en prueba:',round(mean_absolute_error(yte,pred),3),'t/ha')
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    ax[0].hist(X.Humedad_pct.dropna(),bins=12,color=AZUL);ax[0].set(title='¿Cómo se distribuye la humedad?',xlabel='Humedad (%)',ylabel='Número de fincas')
    ax[1].scatter(X.Temperatura_C,y,color=VERDE,alpha=.6);ax[1].set(title='Una relación que conviene explorar',xlabel='Temperatura (°C)',ylabel='Rendimiento simulado (t/ha)');mostrar('03-eda')
    print('NDVI: índice de vegetación, no diagnóstico automático de salud.')
    ndvi=np.array([.65,.72,.58,.75,.60,.81,.70]);print('Desviación estándar NDVI:',round(ndvi.std(),3))
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


## 04. Regresión: predecir una cantidad
 **El problema:** ¿Cómo se relacionan el área de una vivienda y su precio? ¿Qué pasa cuando la relación es curva?

 **Vamos a hacer:** Comparar línea, polinomio, Ridge y Lasso en datos nuevos; mostrar MAE y R².

 **Antes de ejecutar:** Cambia GRADO de 2 a 12. ¿La curva más flexible mejora siempre fuera del entrenamiento?

 **Cómo comprobarlo:** No. Más flexibilidad puede ajustar el ruido. Compara errores de entrenamiento y prueba, sin elegir el modelo mirando repetidamente la prueba.


### Entiende la operación · Grado, penalización y MAE
Los precios se generan con una regla y ruido elegidos. Grado y alpha son configuraciones; los coeficientes los aprende fit. MAE=promedio de |real−predicho| y conserva millones COP en esta simulación. R² compara errores cuadrados con una referencia de media y puede ser negativo. El dibujo muestra entrenamiento y prueba; elegir repetidamente el modelo mirando prueba la convierte en validación.

In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_REPASO_ML:
    from sklearn.linear_model import LinearRegression,Ridge,Lasso
    from sklearn.preprocessing import PolynomialFeatures
    r=np.random.default_rng(SEMILLA)
    area=r.uniform(40,180,100);precio=40+2*area+.008*(area-90)**2+r.normal(0,18,100)
    X=area[:,None];Xtr,Xte,ytr,yte=train_test_split(X,precio,test_size=.25,random_state=SEMILLA)
    GRADO=2
    modelos={'Lineal':LinearRegression(),
             'Polinomio':make_pipeline(PolynomialFeatures(GRADO,include_bias=False),StandardScaler(),LinearRegression()),
             'Ridge':make_pipeline(PolynomialFeatures(GRADO,include_bias=False),StandardScaler(),Ridge(alpha=1)),
             'Lasso':make_pipeline(PolynomialFeatures(GRADO,include_bias=False),StandardScaler(),Lasso(alpha=.5,max_iter=5000))}
    resultados=[];malla=np.linspace(40,180,150)[:,None]
    plt.scatter(Xtr[:,0],ytr,color=GRIS,label='Entrenamiento',alpha=.6)
    plt.scatter(Xte[:,0],yte,color='black',marker='x',label='Prueba')
    for nombre,m in modelos.items():
        m.fit(Xtr,ytr);p=m.predict(Xte)
        resultados.append([nombre,mean_absolute_error(ytr,m.predict(Xtr)),mean_absolute_error(yte,p),r2_score(yte,p)])
        plt.plot(malla[:,0],m.predict(malla),label=nombre)
    display(pd.DataFrame(resultados,columns=['Modelo','MAE entrenamiento','MAE prueba','R² prueba']).round(3))
    plt.xlabel('Área (m²)');plt.ylabel('Precio ilustrativo (millones COP)');plt.legend();plt.title('Predecir un número, no una categoría');mostrar('04-regresion')
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


## 05. Agrupación: encontrar grupos sin etiquetas
 **El problema:** Queremos explorar perfiles de parcelas y concentraciones de puntos de observación.

 **Vamos a hacer:** Comparar K-Means, DBSCAN y OPTICS con gráficos de colores.

 **Antes de ejecutar:** Cambia K de 3 a 4 y EPS de 8 a 3. ¿Los colores son diagnósticos agronómicos?

 **Cómo comprobarlo:** No: son grupos matemáticos. DBSCAN marca puntos no agrupados como −1; no demuestra que sean errores o plagas. OPTICS sí devuelve labels_.


### Entiende la operación · Radio y densidad tienen unidades
Los puntos son coordenadas ficticias en metros. En DBSCAN, EPS=8 significa radio de 8 m, y min_samples=4 cuenta al propio punto. K es número solicitado de grupos en K-Means; no es el número de vecinos de KNN. OPTICS usa una estructura de densidad y sus propios parámetros. Los colores son identificadores locales de cada algoritmo; un mismo color en dos paneles no garantiza las mismas filas.

In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_REPASO_ML:
    from sklearn.cluster import KMeans,DBSCAN,OPTICS
    from sklearn.datasets import make_blobs
    K=3;EPS=8
    puntos,_=make_blobs(n_samples=[35,35,35],centers=[(15,20),(65,70),(80,20)],cluster_std=[2,5,3],random_state=SEMILLA)
    puntos=np.vstack([puntos,[[2,90],[95,90],[45,45]]]) # coordenadas en metros
    modelos={'K-Means':KMeans(n_clusters=K,random_state=SEMILLA,n_init=10),
             'DBSCAN':DBSCAN(eps=EPS,min_samples=4),
             'OPTICS':OPTICS(min_samples=5,xi=.05,min_cluster_size=.1)}
    fig,axs=plt.subplots(1,3,figsize=(12,4))
    for ax,(nombre,m) in zip(axs,modelos.items()):
        etiquetas=m.fit_predict(puntos)
        ax.scatter(puntos[:,0],puntos[:,1],c=etiquetas,cmap='tab10',s=30)
        ax.set(title=nombre,xlabel='X (m)',ylabel='Y (m)');ax.set_aspect('equal')
        print(nombre,'· grupos:',len(set(etiquetas)-{-1}),'· no agrupados:',sum(etiquetas==-1))
    mostrar('05-clustering')
    # Para mezclar rendimiento y humedad, estandariza las variables antes de K-Means.
    # Para coordenadas métricas en el mismo sistema, conservamos metros y EPS también en metros.
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


## 06. Clasificación: asignar una etiqueta
 **El problema:** A partir de color y manchas simulados, queremos clasificar un grano como sano o defectuoso.

 **Vamos a hacer:** Entrenar KNN, SVM, árbol, Naive Bayes y regresión logística con la misma separación y dibujar sus fronteras.

 **Antes de ejecutar:** Cambia VECINOS de 3 a 15. ¿Se vuelve más suave la frontera de KNN?

 **Cómo comprobarlo:** Las fronteras muestran reglas aprendidas. El resultado en prueba estima desempeño en estos datos simulados; no valida inspección de café real.


### Entiende la operación · La etiqueta positiva debe nombrarse
En datos_cafe, 1 significa sano y 0 defectuoso: una puntuación alta de clase 1 no es riesgo de defecto. La regla generadora es inventada; se simulan etiquetas con ruido. Cada modelo aprende con las mismas filas de entrenamiento. El fondo muestra su decisión y los puntos la realidad reservada. La matriz predict_proba tiene columnas en el orden de classes_, no dos predicciones de la misma categoría.

In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_REPASO_ML:
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.naive_bayes import GaussianNB
    from sklearn.linear_model import LogisticRegression
    X,y=datos_cafe();Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
    VECINOS=3
    modelos={'KNN':make_pipeline(StandardScaler(),KNeighborsClassifier(VECINOS)),
             'SVM':make_pipeline(StandardScaler(),SVC(kernel='rbf',C=1)),
             'Árbol':DecisionTreeClassifier(max_depth=3,random_state=SEMILLA),
             'Naive Bayes':GaussianNB(),
             'Logística':make_pipeline(StandardScaler(),LogisticRegression(random_state=SEMILLA))}
    xx,yy=np.meshgrid(np.linspace(0,1,90),np.linspace(0,1,90));malla=pd.DataFrame(np.c_[xx.ravel(),yy.ravel()],columns=X.columns)
    fig,axs=plt.subplots(2,3,figsize=(12,7));filas=[]
    for ax,(nombre,m) in zip(axs.ravel(),modelos.items()):
        m.fit(Xtr,ytr);p=m.predict(Xte);filas.append([nombre,accuracy_score(yte,p)])
        ax.contourf(xx,yy,m.predict(malla).reshape(xx.shape),levels=[-.5,.5,1.5],colors=['#f8e8dc','#e1f1e9'],alpha=.8)
        ax.scatter(Xte.iloc[:,0],Xte.iloc[:,1],c=yte,cmap='RdYlGn',edgecolor='black',s=25)
        ax.set(title=nombre,xlabel='Uniformidad del color',ylabel='Proporción de manchas')
    axs.ravel()[-1].axis('off');mostrar('06-clasificacion')
    display(pd.DataFrame(filas,columns=['Modelo','Accuracy prueba']).round(3))
    print('Ejemplo nuevo: sano=1, defectuoso=0')
    nuevo=pd.DataFrame([[.6,.2]],columns=X.columns)
    print('Probabilidad estimada con logística:',modelos['Logística'].predict_proba(nuevo))
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


## 07. Ensamblados: varios modelos, una predicción
 **El problema:** Un árbol puede cambiar mucho si cambian los datos. ¿Ayuda combinar varios?

 **Vamos a hacer:** Comparar árbol, bagging, Random Forest, AdaBoost, XGBoost y LightGBM usando las mismas fincas y la misma métrica.

 **Antes de ejecutar:** Reduce n_estimators a 5. ¿Todos empeoran necesariamente?

 **Cómo comprobarlo:** No existe un ganador universal. Bagging combina modelos independientes; boosting agrega modelos secuenciales. Esta comparación utiliza regresión, no etiquetas.


### Entiende la operación · Árboles, agregación y referencia
La referencia DummyRegressor usa la media de entrenamiento para todos los casos. Bagging entrena árboles en muestras con reemplazo y combina sus predicciones; Random Forest añade aleatoriedad en las variables. Boosting añade modelos sucesivos. ARBOLES fija la cantidad pedida, no la complejidad aprendida. Las barras son MAE sobre idénticas filas reservadas: más corta significa menor error en t/ha.

In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_REPASO_ML:
    preparar_extension('xgboost','xgboost-cpu')
    preparar_extension('lightgbm','lightgbm')
    from sklearn.ensemble import BaggingRegressor,RandomForestRegressor,AdaBoostRegressor
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.dummy import DummyRegressor
    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor
    X,y=datos_fincas();Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEMILLA)
    ARBOLES=40
    modelos={'Promedio (baseline)':DummyRegressor(),
             'Un árbol':DecisionTreeRegressor(max_depth=4,random_state=SEMILLA),
             'Bagging':BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=4),n_estimators=ARBOLES,random_state=SEMILLA,n_jobs=1),
             'Random Forest':RandomForestRegressor(n_estimators=ARBOLES,max_depth=6,random_state=SEMILLA,n_jobs=1),
             'AdaBoost':AdaBoostRegressor(n_estimators=ARBOLES,random_state=SEMILLA),
             'XGBoost':XGBRegressor(n_estimators=ARBOLES,max_depth=3,learning_rate=.1,random_state=SEMILLA,n_jobs=1),
             'LightGBM':LGBMRegressor(n_estimators=ARBOLES,num_leaves=7,min_child_samples=10,verbosity=-1,random_state=SEMILLA,n_jobs=1)}
    filas=[]
    for nombre,m in modelos.items():
        m.fit(Xtr,ytr);p=m.predict(Xte);filas.append([nombre,mean_absolute_error(yte,p)])
    resultados=pd.DataFrame(filas,columns=['Modelo','MAE prueba']).sort_values('MAE prueba');display(resultados.round(3))
    plt.barh(resultados.Modelo,resultados['MAE prueba'],color=AZUL);plt.xlabel('Error absoluto medio (t/ha) · menor es mejor');plt.title('Comparación en el mismo conjunto de prueba');mostrar('07-ensambles')
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


## 08. Evaluar y explicar: cuánto falla y dónde
 **El problema:** Un detector omite una plaga y también produce una falsa alarma. ¿Cómo contamos esos errores?

 **Vamos a hacer:** Construir matriz de confusión; calcular métricas; validar con grupos y tiempo; descomponer una predicción con SHAP.

 **Antes de ejecutar:** Sube el umbral de .5 a .8 en el ejemplo de probabilidades. ¿Qué pasa con las detecciones?

 **Cómo comprobarlo:** Recall pregunta cuántos positivos reales detectamos; precision pregunta cuántas alertas eran correctas. SHAP explica el modelo, no demuestra causas.


### Entiende la operación · Conteos antes de cocientes
En el ejemplo de plaga, positivo=1. TP: plaga alertada; FN: plaga omitida; FP: alerta sin plaga; TN: sin plaga ni alerta. Precisión=TP/(TP+FP), recall=TP/(TP+FN), F1=2TP/(2TP+FP+FN). El umbral lo elegimos y no cambia las puntuaciones dadas. ROC-AUC usa su orden en varios cortes. GroupKFold separa grupos y TimeSeriesSplit respeta el orden; los grupos de esta demostración son identificadores artificiales. SHAP reparte una predicción respecto de una referencia y no prueba causas.

In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_REPASO_ML:
    from sklearn.metrics import ConfusionMatrixDisplay,precision_score,recall_score,f1_score,roc_auc_score
    verdad=np.array([1,0,1,1,0,0]);prob=np.array([.9,.1,.3,.8,.2,.7]);UMBRAL=.5
    pred=(prob>=UMBRAL).astype(int)
    ConfusionMatrixDisplay.from_predictions(verdad,pred,display_labels=['Sin plaga','Con plaga'],cmap='Blues',colorbar=False)
    plt.title('¿Qué error produce cada casilla?');mostrar('08-metricas')
    print('Accuracy:',accuracy_score(verdad,pred))
    print('Precision:',precision_score(verdad,pred,zero_division=np.nan))
    print('Recall:',recall_score(verdad,pred,zero_division=np.nan))
    print('F1:',f1_score(verdad,pred,zero_division=0),'· ROC-AUC:',roc_auc_score(verdad,prob))
    tp=int(np.sum((pred==1)&(verdad==1)));fn=int(np.sum((pred==0)&(verdad==1)))
    fp=int(np.sum((pred==1)&(verdad==0)));tn=int(np.sum((pred==0)&(verdad==0)))
    print(f'Con umbral {UMBRAL}: TP={tp}, FN={fn}, FP={fp}, TN={tn}')
    print('Precisión = TP/(TP+FP); recall = TP/(TP+FN). NaN significa cociente no definido, no cero.')
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_REPASO_ML:
    from sklearn.model_selection import KFold,GroupKFold,TimeSeriesSplit,cross_val_score
    from sklearn.ensemble import RandomForestRegressor
    X,y=datos_fincas(n=120);grupos=np.repeat(np.arange(12),10)
    reg=RandomForestRegressor(n_estimators=30,max_depth=5,random_state=SEMILLA,n_jobs=1)
    cv=KFold(5,shuffle=True,random_state=SEMILLA)
    mae=-cross_val_score(reg,X,y,cv=cv,scoring='neg_mean_absolute_error')
    print('MAE por partición aleatoria:',np.round(mae,3),'· media:',round(mae.mean(),3))
    # Demostración de separación: si hay varias filas por finca, mantenlas juntas.
    train_g,test_g=next(GroupKFold(3).split(X,y,groups=grupos))
    print('Fincas compartidas entre train y test:',set(grupos[train_g])&set(grupos[test_g]))
    train_t,test_t=next(TimeSeriesSplit(3).split(np.arange(40)))
    print('Ejemplo temporal: entrenar hasta',train_t[-1],'y evaluar desde',test_t[0])
    # SHAP para una predicción de regresión; aquí sí ejecutamos el explicador.
    preparar_extension('shap','shap')
    import shap
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEMILLA)
    reg.fit(Xtr,ytr);explicador=shap.TreeExplainer(reg);explicacion=explicador(Xte.iloc[:1])
    base=float(np.asarray(explicacion.base_values).ravel()[0]);aportes=explicacion.values[0];pred=float(reg.predict(Xte.iloc[:1])[0])
    print('Referencia:',round(base,3),'· aportes:',np.round(aportes,3),'· predicción:',round(pred,3))
    assert np.isclose(base+aportes.sum(),pred,atol=1e-5)
    plt.barh(X.columns,aportes,color=[VERDE if v>0 else AZUL for v in aportes]);plt.axvline(0,color='black',lw=.7);plt.xlabel('Contribución a ESTA predicción (t/ha)');plt.title('SHAP: referencia + contribuciones = predicción');mostrar('08-shap')
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


## 09. De usar una neurona a entrenar una red

**La situación:** ahora distinguimos dos acciones. Primero usamos pesos dados para producir una salida. Después enseñamos a una red con ejemplos simulados de color y manchas.

**Qué significa cada palabra:** una capa reúne unidades de cálculo; una red conecta capas. Entrenar cambia pesos y sesgos usando ejemplos con respuesta conocida. Inferir usa los pesos actuales para responder, sin cambiarlos. En estos datos **1 significa sano y 0 defectuoso**: una salida alta de clase 1 no es riesgo de defecto.

**Ejecuta solo las dos celdas siguientes**, después de la preparación. No necesitan el bloque 06. En la primera verás activaciones, reglas que transforman una suma. En la segunda, una curva de pérdida y las decisiones de una red que sí se entrenó.


### Sigue el ciclo antes de modificar el código

1. La red recibe color y manchas y calcula una puntuación, llamada **logit**. Es un número interno, no un porcentaje.
2. La **pérdida** compara esa puntuación con la respuesta conocida. BCEWithLogitsLoss hace esa comparación para dos categorías; recibe la puntuación sin sigmoide previa.
3. **Retropropagación** calcula cómo contribuye cada ajuste a la pérdida. Eso sucede en backward(). No cambia los pesos por sí sola.
4. El **optimizador**, aquí Adam, usa esa información para cambiarlos en step().

Un **lote** o batch es un grupo de ejemplos por actualización. Una **época** es una pasada por todos los ejemplos de entrenamiento. Una tasa de aprendizaje es un tamaño de paso elegido, no la precisión esperada.

**Qué leer después:** la curva es pérdida de entrenamiento, no porcentaje de aciertos. A la derecha, el fondo muestra puntuación de clase sano y cada punto de prueba muestra su respuesta conocida. Los pesos se entrenan con un grupo y se comprueban con otro. Cambia solo una opción a la vez para comparar.


### 09A · Usar una neurona y comparar activaciones

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Una neurona con parámetros explícitos. No está entrenada: reproduce la guía.
capa=nn.Linear(2,1)
with torch.no_grad():
    capa.weight.copy_(torch.tensor([[2.,-1.]]));capa.bias.copy_(torch.tensor([-.5]))
x=torch.tensor([[.6,.2]])
z=capa(x);p=torch.sigmoid(z)
print('Entradas:',x.tolist(),'· pesos:',capa.weight.detach().tolist(),'· sesgo:',capa.bias.item())
print('Puntuación z:',z.item(),'· salida sigmoid del ejemplo:',p.item())
print('Los parámetros se fijaron a mano: esta salida no es una probabilidad validada de que un grano esté sano.')
valores=torch.linspace(-4,4,150)
fig,axs=plt.subplots(1,4,figsize=(12,3))
for ax,(nombre,f) in zip(axs,[('Identidad',lambda x:x),('Sigmoide',torch.sigmoid),('tanh',torch.tanh),('ReLU',torch.relu)]):
    ax.plot(valores,f(valores),color=AZUL);ax.axhline(0,color=GRIS,lw=.7);ax.set(title=nombre,xlabel='z',ylabel='f(z)')
mostrar('09-activaciones')
print('Softmax [2,1,0]:',torch.softmax(torch.tensor([2.,1.,0.]),dim=0).tolist())


### Después de ejecutar: suma y transformación son pasos distintos

La suma z es 0,5 y su sigmoide es aproximadamente 0,622. ReLU sobre esa misma suma daría 0,5; no deben salir iguales por ser la misma neurona. Los pesos están dados; ninguna de estas cifras prueba que un grano real sea sano.

### 09B · Entrenar una red pequeña

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
X,y=datos_cafe(n=320);Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
escalador=StandardScaler().fit(Xtr)
xt=torch.tensor(escalador.transform(Xtr),dtype=torch.float32);yt=torch.tensor(ytr[:,None],dtype=torch.float32)
xe=torch.tensor(escalador.transform(Xte),dtype=torch.float32)
TASA=.02;TAMANO_LOTE=32;EPOCAS=40
torch.manual_seed(SEMILLA)
red=nn.Sequential(nn.Linear(2,12),nn.ReLU(),nn.Linear(12,8),nn.ReLU(),nn.Linear(8,1))
perdida=nn.BCEWithLogitsLoss() # recibe puntuaciones; no añadir sigmoide antes
opt=torch.optim.Adam(red.parameters(),lr=TASA)
lotes=DataLoader(TensorDataset(xt,yt),batch_size=TAMANO_LOTE,shuffle=True)
historial=[]
for epoca in range(EPOCAS):
    red.train();suma=0
    for bx,by in lotes:
        opt.zero_grad();error=perdida(red(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    historial.append(suma/len(xt))
red.eval()
with torch.no_grad():probs=torch.sigmoid(red(xe)).numpy().ravel()
print('Accuracy en prueba:',accuracy_score(yte,probs>=.5))
fig,axs=plt.subplots(1,2,figsize=(10,4));axs[0].plot(historial,color=AZUL);axs[0].set(title='Aprender: reducir el error',xlabel='Época',ylabel='Pérdida de entrenamiento')
xx,yy=np.meshgrid(np.linspace(0,1,80),np.linspace(0,1,80));malla=pd.DataFrame(np.c_[xx.ravel(),yy.ravel()],columns=X.columns)
with torch.no_grad():mapa=torch.sigmoid(red(torch.tensor(escalador.transform(malla),dtype=torch.float32))).numpy().reshape(xx.shape)
axs[1].contourf(xx,yy,mapa,levels=np.linspace(0,1,11),cmap='RdYlGn');axs[1].scatter(Xte.iloc[:,0],Xte.iloc[:,1],c=yte,cmap='RdYlGn',edgecolor='black',s=20);axs[1].set(title='Predecir: usar los pesos aprendidos',xlabel='Uniformidad del color',ylabel='Proporción de manchas');mostrar('09-red-entrenada')


### Después de ejecutar: tres comprobaciones pequeñas

Identifica cuántos ejemplos quedaron fuera del ajuste. Lee la pérdida al comenzar y al terminar sin convertirla a porcentaje. Luego compara el porcentaje de aciertos en esos ejemplos reservados. Si pruebas otra tasa, registra la comparación; elegir con esa reserva la convierte en validación, no en una prueba final nueva.

## 10. Ver píxeles y distinguir qué respuesta necesitamos

**La situación:** leer un número escrito a mano es una tarea pequeña con respuestas conocidas. Nos permite entender una CNN sin buscar ni subir fotografías. El mismo tipo de operaciones aparece en redes mayores, pero esta red **solo aprende dígitos**, no frutos ni defectos del café.

Un **píxel** es un cuadrito de la imagen. Aquí cada dígito tiene 8 filas y 8 columnas: 64 cuadritos. Guarda brillo de 0 (oscuro) a 16 (claro); dividir por 16 cambia la escala a 0–1. Un **filtro** mira pequeños grupos de píxeles y produce respuestas al desplazarse. Un **mapa** reúne esas respuestas.

La primera celda entrena una CNN y muestra imágenes con respuesta real y calculada. Después muestra **otro filtro fijado a mano**, para leer la operación. La segunda celda dibuja dos objetos y tres tipos de respuesta: etiqueta, cajas y regiones. Son dibujos; no entrenan detección o segmentación.

**Antes de ejecutar:** en «Real 3 → pred. 8», ¿qué significa cada número? Real es la etiqueta conocida; pred. es lo que calculó la red. Una coincidencia se comprueba, no se da por hecha.


### De los cuadritos a la respuesta

La CNN recibe una imagen, aplica filtros de 3×3 y aprende **ocho** mapas. ReLU cambia a cero las respuestas negativas. Una operación de resumen llamada MaxPool toma el máximo de cada zona y reduce los mapas de 8×8 a 4×4. Al final, la red calcula diez puntuaciones, una por dígito; elige la mayor. Esas diez puntuaciones se llaman logits, no diez objetos de la imagen.

**Tip para leer los dibujos:** en la primera figura compara «Real» y «pred.» para cada imagen. En el mapa de bordes, el color representa una respuesta numérica del filtro, no enfermedad ni probabilidad. Ese filtro es fijo y distinto de los ocho que se aprendieron.

**Tip para probar:** cambia solo EPOCAS de 12 a 1 y vuelve a ejecutar la celda entera. Compara las respuestas y la pérdida; no prometas que cada imagen empeorará ni elijas una versión final mirando una y otra vez la misma prueba. Para transferencia continúa en YOLO, secciones 1–6: es cargar otro modelo para otra tarea, no convertir esta CNN de dígitos en detector.


### 10A · Reconocer dígitos con una CNN

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
from torch.utils.data import TensorDataset,DataLoader
D=load_digits();X=D.images.astype('float32')/16;y=D.target
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEMILLA)
xt=torch.tensor(Xtr[:,None]);yt=torch.tensor(ytr,dtype=torch.long);xe=torch.tensor(Xte[:,None])
torch.manual_seed(SEMILLA)
cnn=nn.Sequential(nn.Conv2d(1,8,kernel_size=3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),nn.Linear(8*4*4,10))
opt=torch.optim.Adam(cnn.parameters(),lr=.01);perdida=nn.CrossEntropyLoss();lotes=DataLoader(TensorDataset(xt,yt),batch_size=64,shuffle=True)
EPOCAS=12;hist=[]
for epoca in range(EPOCAS):
    cnn.train();suma=0
    for bx,by in lotes:
        opt.zero_grad();error=perdida(cnn(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(xt))
cnn.eval()
with torch.no_grad():pred=cnn(xe).argmax(1).numpy()
print('CNN · accuracy en prueba:',round(accuracy_score(yte,pred),3))
fig,axs=plt.subplots(2,6,figsize=(11,4))
for i,ax in enumerate(axs.ravel()):
    ax.imshow(Xte[i],cmap='gray');ax.set_title(f'Real {yte[i]} → pred. {pred[i]}',fontsize=9);ax.axis('off')
mostrar('10-cnn')
# Convolución con filtro fijo para visualizar la operación (no es un filtro aprendido).
filtro=torch.tensor([[[[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]]]])
with torch.no_grad():bordes=torch.nn.functional.conv2d(xe[:1],filtro,padding=1).numpy()[0,0]
fig,axs=plt.subplots(1,3,figsize=(10,3));axs[0].imshow(Xte[0],cmap='gray');axs[0].set_title('Entrada: 8×8 píxeles');axs[1].imshow(bordes,cmap='coolwarm');axs[1].set_title('Respuesta del filtro: cambios horizontales');axs[2].plot(hist,color=AZUL);axs[2].set(title='Error de la CNN',xlabel='Época',ylabel='Pérdida');mostrar('10-convolucion')


### Después de ejecutar: localiza un acierto y un error

Mira una imagen y compara la etiqueta Real con pred. Si no hay errores en las doce imágenes dibujadas, no concluyas que no los hay en toda la prueba: el dibujo solo muestra una parte. El mapa de borde corresponde a otro filtro fijo; sus colores no son clases de dígitos.

### 10B · Comparar etiqueta, cajas y regiones

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Ilustración exacta de tres TIPOS de salida, usando objetos sintéticos.
from matplotlib.patches import Rectangle
h,w=60,90;fila,col=np.mgrid[:h,:w]
mascara=((fila-30)**2+(col-25)**2<13**2)|((fila-28)**2+(col-65)**2<10**2)
imagen=np.ones((h,w,3))*.93;imagen[mascara]=[.2,.65,.4]
fig,axs=plt.subplots(1,3,figsize=(11,3))
for ax in axs:ax.axis('off')
axs[0].imshow(imagen);axs[0].set_title('Clasificación: «hay objetos»')
axs[1].imshow(imagen);axs[1].add_patch(Rectangle((12,17),26,26,fill=False,ec='red',lw=2));axs[1].add_patch(Rectangle((55,18),20,20,fill=False,ec='red',lw=2));axs[1].set_title('Detección: ¿dónde están?')
axs[2].imshow(mascara,cmap='Greens');axs[2].set_title('Segmentación: ¿qué píxeles?');mostrar('10-tareas-vision')
print('Estas cajas y máscaras se dibujaron a partir de la geometría: no ejecutan YOLO ni U-Net.')


### Después de ejecutar: ¿qué encargo requiere cada salida?

Para seleccionar imágenes basta una etiqueta global. Para señalar dónde está cada objeto hacen falta cajas. Para delimitar su región se usan píxeles marcados. Aquí hay dos objetos dibujados; el aula usa tres frutos. Cambia la escena, no la diferencia entre las tareas.

## 11. El orden de una secuencia

**DL3 empieza aquí:** una secuencia es una lista en la que importa qué ocurrió antes y qué vino después. En el aula, 10→20→30 cajas y 30→20→10 tienen el mismo total pero diferente recorrido. Aquí usaremos una señal artificial que sube y baja para ver cómo se estima su siguiente lectura.

Ejecuta la sección **11A · Pronosticar la siguiente lectura con LSTM**, después de las preparaciones. Crea sus datos y su red, sin depender del bloque de textos.

La celda de textos anterior es una **ampliación desactivada por defecto**. Bag of Words cuenta palabras; TF-IDF cambia su peso; SVD resume esas listas. No son requisitos para entender ni ejecutar la LSTM de este encuentro.


### Pasado disponible y siguiente lectura

Una **LSTM** es una red que va conservando y actualizando un resumen de lecturas anteriores. Una **ventana** dice cuántas lecturas le damos de entrada: aquí son doce, en orden. Su respuesta es la lectura inmediatamente siguiente. La señal se inventa con una onda y pequeñas variaciones; no son cajas ni un sensor real y no tiene unidad física.

El **corte 180** separa los instantes usados para ajustar de los posteriores, usados para comprobar. En cada instante posterior disponemos de sus doce lecturas reales previas. Por eso es pronóstico de **un paso cada vez**, no producir sesenta pasos futuros sin recibir nuevas observaciones.

**Referencia:** repetir la última lectura conocida. El MAE es el promedio de las distancias entre predicción y valor real; menor indica menos error en esos mismos instantes. En el gráfico, compara las tres líneas en una misma posición del eje horizontal.

**Tip:** no mezcles las fechas al dividir los datos. Antes de cambiar VENTANA, identifica qué doce lecturas se usarían para el primer instante de prueba. Una pérdida pequeña de entrenamiento no garantiza que gane a repetir el último valor.


In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_TEXTO:
    from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.linear_model import LogisticRegression
    textos=['factura pago pendiente','pago factura del proveedor','factura recibida para pago','solicito pago de factura',
            'revisar contrato firmado','firma del contrato pendiente','contrato requiere revisión','solicito revisar contrato',
            'pedido entrega retrasada','entrega del pedido pendiente','pedido llegó incompleto','solicito rastrear pedido']
    etiquetas=['factura']*4+['contrato']*4+['pedido']*4
    v=CountVectorizer();conteos=v.fit_transform(['hay plaga','no hay plaga','plaga hay'])
    display(pd.DataFrame(conteos.toarray(),columns=v.get_feature_names_out(),index=['hay plaga','no hay plaga','plaga hay']))
    Xtr,Xte,ytr,yte=train_test_split(textos,etiquetas,test_size=.25,stratify=etiquetas,random_state=SEMILLA)
    clasificador=make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(max_iter=1000))
    clasificador.fit(Xtr,ytr);print('Accuracy en solo 3 mensajes de prueba:',accuracy_score(yte,clasificador.predict(Xte)))
    print('Nuevo mensaje:',clasificador.predict(['por favor enviar la factura para pago'])[0])
    tfidf=TfidfVectorizer().fit_transform(textos);coords=TruncatedSVD(2,random_state=SEMILLA).fit_transform(tfidf)
    for cat,c in zip(['factura','contrato','pedido'],[AZUL,VERDE,GRIS]):
        idx=np.array(etiquetas)==cat;plt.scatter(coords[idx,0],coords[idx,1],label=cat,s=55)
    plt.legend();plt.xlabel('Componente 1');plt.ylabel('Componente 2');plt.title('Representación reducida de textos (TF-IDF + SVD, no Word2Vec)');mostrar('11-textos')
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


### 11A · Pronosticar la siguiente lectura con LSTM

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Una LSTM real para una secuencia pequeña y artificial.
r=np.random.default_rng(SEMILLA);tiempo=np.arange(240);serie=np.sin(tiempo*.12)+.12*r.normal(size=len(tiempo))
VENTANA=12;CORTE=180
entradas=np.array([serie[i-VENTANA:i] for i in range(VENTANA,len(serie))],dtype='float32')
objetivos=serie[VENTANA:].astype('float32');instante=tiempo[VENTANA:]
train=instante<CORTE;test=~train
xt=torch.tensor(entradas[train,:,None]);yt=torch.tensor(objetivos[train,None]);xe=torch.tensor(entradas[test,:,None])
class PredictorLSTM(nn.Module):
    def __init__(self):
        super().__init__();self.memoria=nn.LSTM(1,12,batch_first=True);self.salida=nn.Linear(12,1)
    def forward(self,x):
        secuencia,_=self.memoria(x);return self.salida(secuencia[:,-1])
torch.manual_seed(SEMILLA);lstm=PredictorLSTM();opt=torch.optim.Adam(lstm.parameters(),lr=.02)
for epoca in range(90):
    opt.zero_grad();error=nn.functional.mse_loss(lstm(xt),yt);error.backward();opt.step()
lstm.eval()
with torch.no_grad():p=lstm(xe).numpy().ravel()
baseline=entradas[test,-1]
print('MAE LSTM:',round(mean_absolute_error(objetivos[test],p),3),'· MAE último valor:',round(mean_absolute_error(objetivos[test],baseline),3))
plt.plot(instante[test],objetivos[test],label='Real simulado',color=GRIS);plt.plot(instante[test],p,label='LSTM',color=AZUL);plt.plot(instante[test],baseline,label='Último valor',color=VERDE,alpha=.6)
plt.legend();plt.xlabel('Tiempo');plt.ylabel('Lectura simulada (sin unidades)');plt.title('Pronóstico de un paso usando la ventana disponible');mostrar('11-lstm')
print('Se evalúa un paso hacia adelante; no es un pronóstico recursivo de toda la serie.')


### Después de ejecutar: compara en los mismos instantes

La línea real es la respuesta reservada; la LSTM y Último valor son dos propuestas para el mismo instante. Busca una zona de subida y otra de bajada y compara cuánto se separan. El MAE resume esas distancias. Repetir el último valor es una referencia que el modelo puede o no mejorar.

## 12. Atención y posición: mezclar no es perder el orden

**Primera pregunta:** tenemos dos cantidades, A=10 y B=20. ¿Cómo cambia una mezcla si tomamos la mitad de cada una o damos más participación a A? Las participaciones son los **pesos de atención** y suman 1, es decir, 100 %.

Una **consulta Q** es la lista de números con la que comparamos. Una **clave K** es la descripción de cada fuente que participa en esa comparación. Un **valor V** es lo que finalmente mezclamos. Aquí los números están dados para explicar la operación; no se aprendieron significados de palabras.

**Segunda pregunta, otro ejemplo:** el Transformer recibe cuatro cantidades y responde si la última es menor que la primera. Una unidad de esa lista se llama **token**. La **posición** indica su lugar para distinguir primero de último. Usamos 0 para “no” y 1 para “sí”. No es un chatbot ni genera texto.

Resuelve antes de ejecutar: 1→2→3→4 da 0; 4→3→2→1 da 1; 2→7→1→3 da 0; 3→1→7→2 da 1. Solo comparamos extremos, no si todo el recorrido baja.

La red incorpora números de posición y conserva las cuatro salidas al aplanarlas. El experimento no aísla el efecto de la posición: no podemos atribuirle por sí sola un acierto. Se quitan secuencias repetidas antes de separar entrenamiento y prueba. Los conteos impresos y el resultado se conocen al ejecutar; no hay un porcentaje garantizado.


### Lee una mezcla antes de leer sus símbolos

Si tomas la mitad de 10 y la mitad de 20, tienes 5+10=15. En la cuenta de atención, una consulta [1,0] da aproximadamente 67 % de participación a A y 33 % a B: la mezcla queda cerca de 13,3, más próxima a 10.

**Vector** significa lista ordenada de números. [1,0] tiene dos posiciones; no es una fracción. **Producto punto** multiplica números del mismo lugar y suma: [1,0] con [1,0] da 1×1 + 0×0 = 1. Se divide por la raíz de 2 porque las listas tienen dos posiciones. **Softmax** convierte esas comparaciones en participaciones positivas que suman 1.

**Tip:** la consulta [0,0] compara igual con A y B. Comprueba que su reparto sea 0,5 y 0,5 y que la salida sea 15. Atención calcula una representación; 13,3 no es una orden de compra ni una cantidad aprendida de ventas.

En el Transformer siguiente sí se ajustan parámetros usando ejemplos de una tarea distinta. Compara sus respuestas con las cuatro etiquetas que resolviste. **Etiqueta** es la respuesta correcta conocida; **predicción** es lo que calcula la red. Una puntuación alta de la red todavía puede equivocarse.


### 12A · Mezclar dos registros con atención

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Dos registros como en el aula: mezclamos cantidades con pesos dados por la comparación.
fuentes=['A: 10','B: 20']
consultas=['Q=[1,0]','Q=[0,1]','Q=[0,0]']
Q=np.array([[1.,0.],[0.,1.],[0.,0.]])
K=np.array([[1.,0.],[0.,1.]])
V=np.array([[10.],[20.]])
puntuaciones=Q@K.T/np.sqrt(2)
exp=np.exp(puntuaciones-puntuaciones.max(axis=1,keepdims=True))
atencion=exp/exp.sum(axis=1,keepdims=True)
mezclas=atencion@V
assert np.allclose(atencion.sum(axis=1),1)
assert np.allclose(atencion[2],[.5,.5]) and np.isclose(mezclas[2,0],15)
for consulta,pesos,mezcla in zip(consultas,atencion,mezclas[:,0]):
    print(consulta,'· participación A/B:',np.round(pesos,4),'· mezcla:',round(float(mezcla),3))
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].imshow(atencion,cmap='Blues',vmin=0,vmax=1)
axs[0].set_xticks(range(2),fuentes);axs[0].set_yticks(range(3),consultas)
for i in range(3):
    for j in range(2):axs[0].text(j,i,f'{atencion[i,j]*100:.1f} %',ha='center',va='center',color='black')
axs[0].set(title='Cuánto participa cada fuente',xlabel='Fuente',ylabel='Consulta dada')
axs[1].bar(consultas,mezclas[:,0],color=[AZUL,VERDE,GRIS]);axs[1].set(title='Resultado de cada mezcla',ylabel='Valor mezclado',ylim=(0,22))
axs[1].axhline(10,ls=':',color='gray');axs[1].axhline(20,ls=':',color='gray')
mostrar('12-atencion')


### Después de ejecutar: comprueba el reparto

Cada fila debe sumar 100 %. Q=[0,0] produce partes iguales y mezcla 15. Q=[1,0] da más participación a A=10 y una salida cercana a 13,3. No se entrenó atención aquí: Q, K y V fueron elegidos para enseñar la operación.

### 12B · Comprobar el orden con un Transformer

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Transformer real, pequeño, entrenado para una tarea de orden.
r=np.random.default_rng(SEMILLA);seq=r.integers(1,8,size=(700,4));seq=seq[seq[:,0]!=seq[:,-1]]
seq=np.unique(seq,axis=0) # reserva de patrones distintos, no solo de filas
y=(seq[:,0]>seq[:,-1]).astype('int64')
Xtr,Xte,ytr,yte=train_test_split(seq,y,test_size=.25,stratify=y,random_state=SEMILLA)
compartidas=set(map(tuple,Xtr)) & set(map(tuple,Xte))
assert not compartidas, 'Entrenamiento y prueba comparten una secuencia.'
print('Secuencias distintas:',len(seq),'· entrenamiento:',len(Xtr),'· prueba:',len(Xte))
print('Clases [0, 1] en entrenamiento:',np.bincount(ytr),'· en prueba:',np.bincount(yte))
class OrdenTransformer(nn.Module):
    def __init__(self):
        super().__init__();self.emb=nn.Embedding(8,16);self.pos=nn.Parameter(torch.randn(1,4,16)*.1)
        capa=nn.TransformerEncoderLayer(d_model=16,nhead=2,dim_feedforward=32,dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(capa,num_layers=1,enable_nested_tensor=False);self.salida=nn.Linear(4*16,2)
    def forward(self,x):
        # La salida por posiciones también conserva orden; no aislamos el efecto de self.pos.
        h=self.encoder(self.emb(x)+self.pos);return self.salida(h.flatten(1))
torch.manual_seed(SEMILLA);transformer=OrdenTransformer();opt=torch.optim.Adam(transformer.parameters(),lr=.005)
loader=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=64,shuffle=True)
hist=[]
for epoca in range(45):
    transformer.train();suma=0
    for bx,by in loader:
        opt.zero_grad();error=nn.functional.cross_entropy(transformer(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(Xtr))
transformer.eval()
with torch.no_grad():
    pred=transformer(torch.tensor(Xte)).argmax(1).numpy()
    ejemplos=torch.tensor([[1,2,3,4],[4,3,2,1],[2,7,1,3],[3,1,7,2]]);prob=torch.softmax(transformer(ejemplos),dim=1).numpy()
print('Accuracy en prueba:',round(accuracy_score(yte,pred),3))
esperadas=(ejemplos[:,0]>ejemplos[:,-1]).numpy().astype(int)
calculadas=prob.argmax(axis=1)
for secuencia,esperada,calculada,p1 in zip(ejemplos.tolist(),esperadas,calculadas,prob[:,1]):
    print(secuencia,'· respuesta conocida:',int(esperada),'· respuesta de la red:',int(calculada),'· puntuación clase 1:',round(float(p1),3))
print('Comparar extremos no pregunta si toda la secuencia baja. Las respuestas de la red pueden fallar.')
plt.plot(hist,color=AZUL);plt.xlabel('Época');plt.ylabel('Pérdida');plt.title('Pérdida de entrenamiento en la tarea de orden');mostrar('12-transformer')


### Después de ejecutar: respuesta correcta y calculada

La etiqueta de los cuatro ejemplos se puede comprobar mirando solo primero y último. La red se comprueba contra esa etiqueta; si falla, el resultado sigue siendo un error aunque su puntuación sea alta. El porcentaje de prueba resume otros patrones reservados y no certifica tareas de lenguaje.

## 13. Ampliación fuera de los encuentros: reconstruir y generar
 **El problema:** Una imagen está dañada por ruido. ¿Podemos aprender a reconstruirla? ¿Qué significa aprender a quitar ruido para generar?

 **Vamos a hacer:** Entrenar un autoencoder sobre dígitos y un modelo de difusión pequeño sobre puntos 2D.

 **Antes de ejecutar:** Aumenta RUIDO de .35 a .5. ¿La reconstrucción recupera todos los detalles?

 **Cómo comprobarlo:** No: puede suavizar o inventar detalles. Un autoencoder reconstruye; la difusión entrenada genera muestras. Los puntos 2D no son imágenes fotográficas. La gráfica permite comparar separación y dispersión; no certifica que el modelo haya recuperado la distribución. Con el calendario de ruido elegido queda señal al final, así que iniciar la generación desde ruido normal es una aproximación.

**No se necesita para los encuentros de DL.** EJECUTAR_GENERACION permanece en False; las dos celdas se omiten sin entrenar. No actives esta ampliación para resolver las tareas de visión, atención o detección.


### Entiende la operación · Reconstrucción y generación son tareas diferentes
El autoencoder recibe una imagen dañada y aprende contra la limpia; la pérdida MSE promedia diferencias cuadradas de píxeles. Un detalle que parece plausible no demuestra fidelidad. En difusión se elige un calendario beta: x_t=√alpha_bar_t·x_0 + √(1−alpha_bar_t)·ruido. La red aprende a estimar ese ruido. PASOS es longitud del proceso y ITERACIONES cantidad de ajustes: no son lo mismo. Los ejes de los puntos 2D no tienen unidades físicas.

In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_GENERACION:
    # Autoencoder de eliminación de ruido: aprende entrada dañada → objetivo limpio.
    D=load_digits();X=D.data.astype('float32')/16
    Xtr,Xte=train_test_split(X,test_size=.25,random_state=SEMILLA)
    xt=torch.tensor(Xtr);xe=torch.tensor(Xte)
    torch.manual_seed(SEMILLA);ae=nn.Sequential(nn.Linear(64,24),nn.ReLU(),nn.Linear(24,16),nn.ReLU(),nn.Linear(16,24),nn.ReLU(),nn.Linear(24,64),nn.Sigmoid())
    opt=torch.optim.Adam(ae.parameters(),lr=.01);RUIDO=.35
    loader=DataLoader(TensorDataset(xt),batch_size=64,shuffle=True)
    for epoca in range(100):
        for (limpio,) in loader:
            dañado=torch.clamp(limpio+RUIDO*torch.randn_like(limpio),0,1)
            opt.zero_grad();error=nn.functional.mse_loss(ae(dañado),limpio);error.backward();opt.step()
    ae.eval();torch.manual_seed(77);ruidoso=torch.clamp(xe[:6]+RUIDO*torch.randn_like(xe[:6]),0,1)
    with torch.no_grad():recon=ae(ruidoso)
    fig,axs=plt.subplots(3,6,figsize=(10,5))
    for fila,(datos,titulo) in enumerate([(xe[:6],'Original'),(ruidoso,'Con ruido'),(recon,'Reconstrucción')]):
        for i,ax in enumerate(axs[fila]):ax.imshow(datos[i].numpy().reshape(8,8),cmap='gray',vmin=0,vmax=1);ax.axis('off');ax.set_title(titulo if i==0 else '',fontsize=10)
    mostrar('13-autoencoder')
    print('MSE ruidoso:',nn.functional.mse_loss(ruidoso,xe[:6]).item(),'· MSE reconstruido:',nn.functional.mse_loss(recon,xe[:6]).item())
    print('No es SAM, DINOv2 ni un Masked Autoencoder de Transformers; es un autoencoder pequeño entrenado aquí.')
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


In [ ]:
# Ampliación fuera del recorrido principal; desactivada por defecto.
if EJECUTAR_GENERACION:
    # Difusión DDPM en 2D: añadir ruido y aprender a predecirlo.
    # Dos grupos artificiales. No requiere descargar un modelo grande.
    torch.manual_seed(SEMILLA);n=600
    centros=torch.tensor([[-1.5,0.],[1.5,0.]])
    limpios=centros[torch.arange(n)%2]+.3*torch.randn(n,2)
    PASOS=50;beta=torch.linspace(.0005,.08,PASOS);alpha=1-beta;acumulado=torch.cumprod(alpha,0)
    class QuitaRuido(nn.Module):
        def __init__(self):
            super().__init__();self.red=nn.Sequential(nn.Linear(3,48),nn.SiLU(),nn.Linear(48,48),nn.SiLU(),nn.Linear(48,2))
        def forward(self,x,t):return self.red(torch.cat([x,t[:,None].float()/(PASOS-1)],dim=1))
    modelo_ruido=QuitaRuido();opt=torch.optim.Adam(modelo_ruido.parameters(),lr=.002)
    ITERACIONES=900
    for i in range(ITERACIONES):
        x0=limpios[torch.randint(0,n,(128,))];t=torch.randint(0,PASOS,(128,));ruido=torch.randn_like(x0)
        abar=acumulado[t,None];xt=abar.sqrt()*x0+(1-abar).sqrt()*ruido
        opt.zero_grad();error=nn.functional.mse_loss(modelo_ruido(xt,t),ruido);error.backward();opt.step()
    modelo_ruido.eval();x=torch.randn(600,2)
    with torch.no_grad():
        for paso in reversed(range(PASOS)):
            t=torch.full((len(x),),paso,dtype=torch.long);eps=modelo_ruido(x,t)
            media=(x-beta[paso]/torch.sqrt(1-acumulado[paso])*eps)/torch.sqrt(alpha[paso])
            if paso>0:
                var=beta[paso]*(1-acumulado[paso-1])/(1-acumulado[paso]);x=media+var.sqrt()*torch.randn_like(x)
            else:x=media
    fig,axs=plt.subplots(1,3,figsize=(11,3))
    ruido=torch.randn_like(limpios)
    for ax,(datos,titulo) in zip(axs,[(limpios,'Distribución de entrenamiento'),(ruido,'Inicio: ruido'),(x,'Muestras generadas')]):
        ax.scatter(datos[:,0],datos[:,1],s=8,color=AZUL,alpha=.4);ax.set(title=titulo,xlim=(-4,4),ylim=(-4,4),xlabel='Eje 1',ylabel='Eje 2');ax.set_aspect('equal')
    mostrar('13-difusion')
    print('Compara separación y dispersión: esta inspección visual no certifica recuperación de la distribución.')
    print('Valor final de alpha_bar_T:',round(float(acumulado[-1]),4))
    print('Queda señal al final; iniciar la generación desde ruido normal es una aproximación de este ejemplo pequeño.')
else:
    print('Ampliación omitida. Continúa con la ruta de tu encuentro; no necesitas activarla.')


### Referencias externas, fuera del recorrido de los encuentros

Los cuatro encuentros se resuelven con sus celdas principales y el cuaderno YOLO indicado. No requieren instalar ni ejecutar SAM, DINOv2, modelos de difusión de imágenes o modelos de lenguaje. Estas referencias son lectura opcional para otro momento; no sustituyen la práctica de las dos asignaturas.

Para ampliar por iniciativa propia, consulta las instrucciones y condiciones de los proyectos originales. No cambies a recursos de pago para completar este laboratorio. El pequeño Transformer del bloque 12 no es un modelo de conversación.


## 14. Una puntuación no decide por nosotros

**La situación:** debemos decidir qué granos revisar y cuáles aceptar sin revisión. Son diez granos ficticios con respuesta conocida: 1 significa **defectuoso**, a diferencia de la red del bloque 09, donde 1 significaba sano.

Una **política** es la regla que convierte puntuación en acción. El **umbral** es un límite elegido: aquí revisamos si la puntuación es mayor o igual a él. Los **costes** son puntos inventados para comparar consecuencias, no precios reales ni ganancias demostradas.

Ejecuta después de la preparación; no depende del bloque 09 ni de sus predicciones. Compara revisar todos con revisar según distintos límites. La primera decisión no debería ser elegir 0,50 por costumbre, sino explicar qué consecuencia estás comparando.


### Haz una cuenta antes de mirar la figura

Con límite 0,50 se revisan cuatro granos y se deja pasar un defecto. Revisar cuesta 1 por grano; omitir un defecto cuesta 8. Por tanto, 4×1 + 1×8 = **12 puntos**. Con límite 0,30 se revisan seis y no se omite ningún defecto: **6 puntos**. Revisar los diez costaría **10**.

Las puntuaciones y respuestas están dadas: **no vienen de la red entrenada**. Suponemos revisión perfecta para aislar la comparación. En una operación real habría que estudiar su capacidad y sus errores.

**Tip:** cambia solo COSTO_DEFECTO_ACEPTADO a 1 y ejecuta de nuevo. Comprueba que cambie la comparación aunque los diez granos y sus puntuaciones no cambien. La figura muestra puntos de coste; un punto bajo no demuestra ahorro comercial.


### 14A · Comparar reglas de revisión

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Positivo = defectuoso. Una puntuación alta dispara revisión manual.
real_defecto=np.array([0,1,0,1,1,0,0,1,0,0])
p_defecto=np.array([.1,.8,.2,.4,.9,.1,.6,.7,.2,.3])
COSTO_REVISION=1.;COSTO_DEFECTO_ACEPTADO=8.
filas=[]
for umbral in [.3,.5,.7,.9]:
    revisar=p_defecto>=umbral
    costo=revisar.sum()*COSTO_REVISION+np.sum((~revisar)&(real_defecto==1))*COSTO_DEFECTO_ACEPTADO
    filas.append([umbral,revisar.sum(),np.sum((~revisar)&(real_defecto==1)),costo])
resultado=pd.DataFrame(filas,columns=['Umbral','Granos revisados','Defectos omitidos','Costo ilustrativo']);display(resultado)
plt.plot(resultado.Umbral,resultado['Costo ilustrativo'],'o-',color=AZUL);plt.axhline(len(real_defecto)*COSTO_REVISION,color=VERDE,ls='--',label='Revisar todos');plt.xlabel('Umbral para revisar');plt.ylabel('Puntos de coste supuestos');plt.title('La decisión de negocio también tiene un parámetro');plt.legend();mostrar('14-impacto')
print('Supuesto del ejercicio: la revisión manual detecta todos los defectos. En un proyecto real, esa revisión también debe evaluarse.')


### Después de ejecutar: justifica la regla, no solo el mínimo

Con los costes iniciales, 0,30 obtiene 6 puntos y revisar todos 10. Explica qué revisión adicional evita qué omisión. Si omitir costara 1, los límites 0,70 y 0,90 empatan en 4 puntos: una decisión necesita también considerar capacidad y consecuencias, no solo una cifra. Cambiar un supuesto puede cambiar la opción preferida.

## Para cerrar: explica, no solo ejecuta
 Para la conversación de clase, revisa un gráfico, una métrica de prueba y una comparación con una solución simple. Explica qué cambiaría si los datos vinieran de otra finca, otro año u otra cámara. Distingue lo que el modelo calcula de lo que tú decides hacer con el resultado.

 ### Fuentes primarias y rutas de profundización
 - [MIT 6.S191](https://introtodeeplearning.com/): referencia para combinar concepto, laboratorio y proyecto. Este cuaderno es original para UTP, no un material oficial del MIT.
 - [Stanford CS231n](https://cs231n.github.io/): fundamentos visuales de redes y visión.
 - [scikit-learn: evitar fugas de datos](https://scikit-learn.org/stable/common_pitfalls.html).
 - [scikit-learn: agrupación](https://scikit-learn.org/stable/modules/clustering.html).
 - [PyTorch: entrenamiento](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html).
 - [XGBoost](https://xgboost.readthedocs.io/en/stable/python/python_api.html), [LightGBM](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRegressor.html), [SHAP](https://shap.readthedocs.io/en/stable/generated/shap.TreeExplainer.html).
 - [Attention Is All You Need](https://arxiv.org/abs/1706.03762) y [DDPM](https://arxiv.org/abs/2006.11239).

 **Reproducibilidad:** semilla 42. La celda de preparación imprime la versión de PyTorch; las métricas pueden variar entre versiones. Las métricas y los PNG se generan al ejecutar las celdas. El archivo de referencia no incluye salidas guardadas ni garantiza un nivel de accuracy.
